In [1]:
import os

# Search ALL of /kaggle/input and print every folder (not files)
print("=== Full directory tree ===")
for root, dirs, files in os.walk('/kaggle/input'):
    # Skip printing image files, just show folders
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")

=== Full directory tree ===
input/
  datasets/
    paultimothymooney/
      chest-xray-pneumonia/
        chest_xray/
          chest_xray/
            val/
              PNEUMONIA/
              NORMAL/
            test/
              PNEUMONIA/
              NORMAL/
            train/
              PNEUMONIA/
              NORMAL/
          __MACOSX/
            chest_xray/
              val/
                PNEUMONIA/
                NORMAL/
              test/
                PNEUMONIA/
                NORMAL/
              train/
                PNEUMONIA/
                NORMAL/
          val/
            PNEUMONIA/
            NORMAL/
          test/
            PNEUMONIA/
            NORMAL/
          train/
            PNEUMONIA/
            NORMAL/


In [2]:
base = '/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray'

train_dir = os.path.join(base, 'train')
val_dir   = os.path.join(base, 'val')
test_dir  = os.path.join(base, 'test')

print(f"train → {train_dir}  exists: {os.path.exists(train_dir)}")
print(f"val   → {val_dir}    exists: {os.path.exists(val_dir)}")
print(f"test  → {test_dir}   exists: {os.path.exists(test_dir)}")

train → /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/train  exists: True
val   → /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/val    exists: True
test  → /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/test   exists: True


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import os

# ──────────────────────────────────────────────
# 1. PATHS  (confirmed from your directory tree)
# ──────────────────────────────────────────────
base      = '/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray'
train_dir = os.path.join(base, 'train')
val_dir   = os.path.join(base, 'val')
test_dir  = os.path.join(base, 'test')

# ──────────────────────────────────────────────
# 2. TRANSFORMS
# ──────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

train_data = datasets.ImageFolder(train_dir, transform=train_tf)
test_data  = datasets.ImageFolder(test_dir,  transform=eval_tf)

train_loader = torch.utils.data.DataLoader(
    train_data, batch_size=32, shuffle=True,  num_workers=2)
test_loader  = torch.utils.data.DataLoader(
    test_data,  batch_size=32, shuffle=False, num_workers=2)

print(f"Classes : {train_data.classes}")
print(f"Train   : {len(train_data)} images")
print(f"Test    : {len(test_data)}  images")

# ──────────────────────────────────────────────
# 3. MODEL  (ResNet50, frozen backbone)
# ──────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 2)
)
model = model.to(device)

# ──────────────────────────────────────────────
# 4. TRAINING
# ──────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.5)

for epoch in range(3):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total   += labels.size(0)

    scheduler.step()
    print(f"Epoch [{epoch+1}/3]  "
          f"Loss: {running_loss/total:.4f}  "
          f"Acc: {100*correct/total:.2f}%")

# ──────────────────────────────────────────────
# 5. TEST EVALUATION
# ──────────────────────────────────────────────
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        correct += (model(images).argmax(1) == labels).sum().item()
        total   += labels.size(0)

test_acc = 100 * correct / total
print(f"\n✅ Test Accuracy: {test_acc:.2f}%")

# ──────────────────────────────────────────────
# 6. SAVE
# ──────────────────────────────────────────────
save_path = '/kaggle/working/pneumonia_resnet50.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'classes':          train_data.classes,   # ['NORMAL', 'PNEUMONIA']
    'test_accuracy':    test_acc,
}, save_path)
print(f"Model saved → {save_path}")

Classes : ['NORMAL', 'PNEUMONIA']
Train   : 5216 images
Test    : 624  images
Device  : cuda
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 192MB/s] 


Epoch [1/3]  Loss: 0.2495  Acc: 89.36%
Epoch [2/3]  Loss: 0.1678  Acc: 93.29%
Epoch [3/3]  Loss: 0.1481  Acc: 94.17%

✅ Test Accuracy: 88.62%
Model saved → /kaggle/working/pneumonia_resnet50.pth
